## 1. Dataset Loading and Inspection

In this section, we load the competition dataset and inspect the structure, columns, row counts, missing values, and basic characteristics of the three sources and ground truth.

In [10]:
import pandas as pd

base = r"E:\Amazon-ML-Challenge-2026\data\student_resource"

train_source1 = pd.read_csv(f"{base}\\dataset\\train\\train_source1.tsv", sep="\t")
train_source2 = pd.read_csv(f"{base}\\dataset\\train\\train_source2.tsv", sep="\t")
train_source3 = pd.read_csv(f"{base}\\dataset\\train\\train_source3.tsv", sep="\t")
train_ground_truth = pd.read_csv(f"{base}\\dataset\\train\\train_ground_truth.tsv", sep="\t")

test_source1 = pd.read_csv(f"{base}\\dataset\\test\\test_source1.tsv", sep="\t")
test_source2 = pd.read_csv(f"{base}\\dataset\\test\\test_source2.tsv", sep="\t")
test_source3 = pd.read_csv(f"{base}\\dataset\\test\\test_source3.tsv", sep="\t")

print("Loaded OK")

Loaded OK


In [11]:
for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3),
    ("train_ground_truth", train_ground_truth),
    ("test_source1", test_source1),
    ("test_source2", test_source2),
    ("test_source3", test_source3),
]:
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("missing values:\n", df.isna().sum())


=== train_source1 ===
shape: (2206821, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id           0
business_name       0
business_address    0
country             0
dtype: int64

=== train_source2 ===
shape: (5034616, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name            2
business_address    168967
country                  0
dtype: int64

=== train_source3 ===
shape: (5285603, 4)
columns: ['entity_id', 'business_name', 'business_address', 'country']
missing values:
 entity_id                0
business_name           13
business_address    175916
country                  0
dtype: int64

=== train_ground_truth ===
shape: (2206821, 2)
columns: ['source1_entity_id', 'matched_entity_ids']
missing values:
 source1_entity_id          0
matched_entity_ids    123247
dtype: int64

=== test_source1 ===
shape: (1732544, 4)
columns: ['entity_id', 'business_

## 2. Ground Truth and Dataset Distribution

Before designing the matching system, we inspect the distribution of
matches and countries in the training data.

In [12]:
train_source1.head(5)

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [13]:
train_source2.head(5)

,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


In [14]:
train_ground_truth.head(10)

,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [15]:
# Confirm the missing ground truth rows really are "no match" (empty string vs NaN)
print(train_ground_truth['matched_entity_ids'].isna().sum())
print((train_ground_truth['matched_entity_ids'] == '').sum())

# How many matches does a typical Source 1 entity have? (only for non-missing rows)
match_counts = train_ground_truth['matched_entity_ids'].dropna().apply(lambda x: len(x.split(',')))
print(match_counts.describe())

# Country distribution across sources
print(train_source1['country'].value_counts())
print(train_source2['country'].value_counts())

123247
0
count    2.083574e+06
mean     3.665992e+00
std      1.526294e+00
min      1.000000e+00
25%      3.000000e+00
50%      4.000000e+00
75%      5.000000e+00
max      1.100000e+01
Name: matched_entity_ids, dtype: float64
country
US       1323633
India     883188
Name: count, dtype: int64
country
US       3016817
India    2017799
Name: count, dtype: int64


In [16]:
import re

def has_devanagari(text):
    if pd.isna(text):
        return False
    return bool(re.search(r'[\u0900-\u097F]', text))

for name, df in [
    ("train_source1", train_source1),
    ("train_source2", train_source2),
    ("train_source3", train_source3)
]:
    n = df['business_name'].apply(has_devanagari).sum()
    print(f"{name}: {n} / {len(df)} names contain Devanagari script ({n/len(df)*100:.2f}%)")

train_source1: 0 / 2206821 names contain Devanagari script (0.00%)
train_source2: 269424 / 5034616 names contain Devanagari script (5.35%)
train_source3: 158003 / 5285603 names contain Devanagari script (2.99%)


## 3. Train-Validation Split

The training data contains Source 1 records together with their ground-truth
matches from Source 2 and Source 3.

To evaluate our matching approach before applying it to the unseen test data,
we split the Source 1 entities into two groups:

- **80% training/development set** — used to develop the baseline approach.
- **20% validation set** — used to evaluate the approach on unseen Source 1 entities.

The split is random and uses a fixed random seed (`random_state=42`) so that
the same validation set can be reproduced across experiments.

Source 2 and Source 3 are not split at this stage because they serve as the
candidate databases against which Source 1 entities are matched.

In [17]:
from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    train_source1['entity_id'],
    test_size=0.20,
    random_state=42
)

print("Training Source 1 entities:", len(train_ids))
print("Validation Source 1 entities:", len(val_ids))

# Sanity check: no overlap, and counts add up
assert len(set(train_ids) & set(val_ids)) == 0
assert len(train_ids) + len(val_ids) == len(train_source1)
print("Split check passed.")

Training Source 1 entities: 1765456
Validation Source 1 entities: 441365
Split check passed.


In [18]:
train_source1_split = train_source1[
    train_source1['entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_source1 = train_source1[
    train_source1['entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

train_ground_truth_split = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(train_ids)
].copy().reset_index(drop=True)

val_ground_truth = train_ground_truth[
    train_ground_truth['source1_entity_id'].isin(val_ids)
].copy().reset_index(drop=True)

print("Train S1:", train_source1_split.shape)
print("Validation S1:", val_source1.shape)
print("Train ground truth:", train_ground_truth_split.shape)
print("Validation ground truth:", val_ground_truth.shape)

Train S1: (1765456, 4)
Validation S1: (441365, 4)
Train ground truth: (1765456, 2)
Validation ground truth: (441365, 2)


## 4. Baseline Matching Approach

We now develop a simple entity-resolution baseline using the training data
and evaluate its predictions on the held-out validation set.

The baseline will first focus on high-confidence matching rules and
candidate generation before introducing more complex matching methods.

### 4.1 Validation Evaluation

The competition evaluates predictions using macro-averaged F₀.₅ across
Source 1 entities. Since precision is weighted more heavily than recall,
the evaluation function measures the overlap between the predicted and
ground-truth match sets for each Source 1 entity.

This function will be used to evaluate our baseline on the held-out
validation set before generating test predictions.

In [19]:
def parse_ids(cell):
    """Convert a comma-separated ID string (or NaN) into a set of IDs."""
    if pd.isna(cell) or cell == '':
        return set()
    return set(cell.split(','))


def f_beta_score(true_set, pred_set, beta=0.5):
    """Compute F_beta for one entity given its true and predicted match sets."""
    # Singleton case: no true matches
    if len(true_set) == 0:
        return 1.0 if len(pred_set) == 0 else 0.0

    # We have true matches but predicted nothing
    if len(pred_set) == 0:
        return 0.0

    tp = len(true_set & pred_set)
    precision = tp / len(pred_set)
    recall = tp / len(true_set)

    if precision == 0 and recall == 0:
        return 0.0

    beta_sq = beta ** 2
    return (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)


def evaluate_predictions(ground_truth_df, predictions_df):
    """
    ground_truth_df: columns [source1_entity_id, matched_entity_ids]
    predictions_df:  columns [source1_entity_id, matched_entity_ids]
    Returns macro-averaged F_0.5 across all entities in ground_truth_df.
    """
    gt = ground_truth_df.set_index('source1_entity_id')['matched_entity_ids']
    pred = predictions_df.set_index('source1_entity_id')['matched_entity_ids']

    scores = []
    for entity_id in gt.index:
        true_set = parse_ids(gt.loc[entity_id])
        # If we didn't predict anything for this entity at all, treat as empty
        pred_cell = pred.loc[entity_id] if entity_id in pred.index else None
        pred_set = parse_ids(pred_cell)
        scores.append(f_beta_score(true_set, pred_set))

    return sum(scores) / len(scores)

In [20]:
import pandas as pd

test_gt = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S3-00812']
})
test_pred = pd.DataFrame({
    'source1_entity_id': ['S1-00001'],
    'matched_entity_ids': ['S2-00047,S2-00193,S3-00812']
})

score = evaluate_predictions(test_gt, test_pred)
print(score)  # PDF says this should be 0.714

0.7142857142857142


## Evaluation Function — Verification

Before using the evaluation function on our real validation data, we tested it
with a small manually constructed example.

The ground truth contained two correct matches, while the prediction contained
those two matches plus one incorrect match. The resulting F_0.5 score was
approximately 0.7143, confirming that the evaluation function penalizes the
extra false match while giving credit for the two correct matches.

This is only a test of the evaluation code. No model has been trained and no
real validation score has been calculated yet.

### 4.2 Name Normalization for Blocking

Before generating candidate pairs, we first normalize business names so that
records representing the same business are less likely to be separated by
simple formatting differences.

The baseline normalization performs four operations:

- Converts the business name to lowercase.
- Replaces punctuation with spaces.
- Removes selected common legal suffixes such as `Inc`, `Ltd`, `LLC`, and
  `Corporation`.
- Collapses whitespace and reconstructs the normalized name.

This normalized representation will later be used to construct a blocking key.
The purpose of this first experiment is to establish a simple baseline before
testing more sophisticated blocking and matching strategies.

In [21]:
import re

def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    
    # Strip punctuation (& . , - etc.), keep letters/digits/spaces
    name = re.sub(r'[^\w\s]', ' ', name)
    
    # Remove common legal-suffix noise words
    stopwords = {'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
                 'corp', 'corporation', 'llc', 'llp', 'co', 'company'}
    tokens = [t for t in name.split() if t not in stopwords]
    
    # Collapse whitespace and return
    return ' '.join(tokens).strip()

In [22]:
sample_names = (
    train_source1['business_name'].head(10).tolist()
    + train_source2['business_name'].head(10).tolist()
    + train_source3['business_name'].head(10).tolist()
)

for n in sample_names:
    print(repr(n), "->", repr(normalize_name(n)))

"Orelee's Barbershop" -> 'orelee s barbershop'
'Prime Money' -> 'prime money'
'B+ Retail Inc' -> 'b retail'
'Christ Chapel' -> 'christ chapel'
'Prabhav Business Center' -> 'prabhav business center'
'Custom Wealth Services LLC' -> 'custom wealth services'
'Consulting Nyasa Nursing Private Limited' -> 'consulting nyasa nursing'
'Nexus Anchor Rain' -> 'nexus anchor rain'
'Moore Bitwise Inc' -> 'moore bitwise'
'Dermatology Green Medicine' -> 'dermatology green medicine'
'राम मार्केटिंग प्राइवेट लिमिटेड' -> 'र म म र क ट ग प र इव ट ल म ट ड'
'-- Holloway Peak Inc Seafood' -> 'holloway peak seafood'
'आदित्य प्रॉपर्टीज एलएलपी' -> 'आद त य प र पर ट ज एलएलप'
'Summit Inc' -> 'summit'
'Delta Tetlecommunication Inc' -> 'delta tetlecommunication'
'Lee and Lawson' -> 'lee and lawson'
'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com' -> 'shivshakti vidyalaya vidyalaya overseas www shivshakti com'
'Shree Infracon Private Ltd' -> 'shree infracon'
'Chavira Platinum Chimera LLC' -> 

### Normalization Sanity Check — Baseline

The normalization function was tested on a small sample of real business names
from all three sources.

For Latin-script names, the baseline behaves as expected in many cases. It
removes capitalization differences, punctuation, and common legal suffixes
while preserving the main business name.

However, the sample also revealed an important issue with non-Latin text.
Devanagari business names are being corrupted during punctuation removal. For
example, a Hindi name such as `राम मार्केटिंग प्राइवेट लिमिटेड` is transformed
into separated characters rather than preserving the original words.

This happens because the current regular expression does not correctly preserve
the Unicode combining characters used in Devanagari text.

Therefore, this version of the normalization function is not suitable as the
final baseline. We will create a new normalization experiment that handles
Unicode text more safely while preserving the useful normalization steps
identified here.

### 4.3 Experiment — Unicode-Safe Name Normalization

The previous normalization experiment showed that punctuation and legal-suffix
normalization works for many Latin-script names but breaks Devanagari text:
the `\w` regex character class does not recognize Devanagari vowel marks
(matras) as word characters, so they were stripped and words were shredded
into individual base letters.

In this experiment, we modify only the punctuation-handling step. Instead of
matching on `\w`, we remove characters using their Unicode general category
(punctuation categories only), which preserves letters and combining marks
across scripts while still stripping punctuation and normalizing whitespace.

The revised function is tested on the same sample as before so the two
normalization approaches can be compared directly.

In [23]:
import unicodedata

def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    
    # Remove only true punctuation characters (Unicode category starting with 'P'),
    # keep letters, combining marks (needed for Devanagari), digits, and spaces
    name = ''.join(' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch for ch in name)
    
    stopwords = {'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
                 'corp', 'corporation', 'llc', 'llp', 'co', 'company'}
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [24]:
sample_names = (
    train_source1['business_name'].head(10).tolist()
    + train_source2['business_name'].head(10).tolist()
    + train_source3['business_name'].head(10).tolist()
)

for n in sample_names:
    print(repr(n), "->", repr(normalize_name(n)))

"Orelee's Barbershop" -> 'orelee s barbershop'
'Prime Money' -> 'prime money'
'B+ Retail Inc' -> 'b retail'
'Christ Chapel' -> 'christ chapel'
'Prabhav Business Center' -> 'prabhav business center'
'Custom Wealth Services LLC' -> 'custom wealth services'
'Consulting Nyasa Nursing Private Limited' -> 'consulting nyasa nursing'
'Nexus Anchor Rain' -> 'nexus anchor rain'
'Moore Bitwise Inc' -> 'moore bitwise'
'Dermatology Green Medicine' -> 'dermatology green medicine'
'राम मार्केटिंग प्राइवेट लिमिटेड' -> 'राम मार्केटिंग प्राइवेट लिमिटेड'
'-- Holloway Peak Inc Seafood' -> 'holloway peak seafood'
'आदित्य प्रॉपर्टीज एलएलपी' -> 'आदित्य प्रॉपर्टीज एलएलपी'
'Summit Inc' -> 'summit'
'Delta Tetlecommunication Inc' -> 'delta tetlecommunication'
'Lee and Lawson' -> 'lee and lawson'
'SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPORATION | www.shivshakti.com' -> 'shivshakti vidyalaya vidyalaya overseas www shivshakti com'
'Shree Infracon Private Ltd' -> 'shree infracon'
'Chavira Platinum Chimera LLC' -

### 4.4 Blocking Key and Candidate Generation

After normalizing business names, we construct a simple blocking key to reduce
the number of Source 2 and Source 3 records that must be considered for each
Source 1 entity.

For this first blocking experiment, the key consists of the country and the
first token of the normalized business name. Using the entire normalized name
would be more restrictive and could discard true matches when names contain
typos or other variations. Using only the first token produces larger but more
forgiving candidate groups, while the country helps limit unrelated records.

We build lookup indexes for Source 2 and Source 3 so that candidates can be
retrieved efficiently without comparing every Source 1 entity against every
record in the other sources.

The blocking stage is evaluated on the held-out validation set using:
- candidate recall: the fraction of known true matches that survive blocking;
- average candidates per Source 1 entity: the size of the resulting candidate
  sets.

Candidate recall represents the recall ceiling of the downstream matching
stage: a true match that is removed during blocking cannot be recovered later.

In [25]:
def first_token(normalized_name):
    tokens = normalized_name.split()
    return tokens[0] if tokens else ""

def build_block_key(df):
    """Adds norm_name and block_key columns to a dataframe with business_name/country."""
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['block_key'] = df['country'] + '||' + df['norm_name'].apply(first_token)
    return df

In [26]:
train_source2_keyed = build_block_key(train_source2)
train_source3_keyed = build_block_key(train_source3)
val_source1_keyed = build_block_key(val_source1)

print("Done building keys.")

Done building keys.


In [27]:
def build_index(df_keyed):
    """Returns dict: block_key -> list of entity_ids"""
    return df_keyed.groupby('block_key')['entity_id'].apply(list).to_dict()

index_source2 = build_index(train_source2_keyed)
index_source3 = build_index(train_source3_keyed)

print("Number of distinct keys in Source 2 index:", len(index_source2))
print("Number of distinct keys in Source 3 index:", len(index_source3))

Number of distinct keys in Source 2 index: 363912
Number of distinct keys in Source 3 index: 412262


In [28]:
def get_candidates(block_key):
    c2 = index_source2.get(block_key, [])
    c3 = index_source3.get(block_key, [])
    return set(c2) | set(c3)

val_gt_lookup = val_ground_truth.set_index('source1_entity_id')['matched_entity_ids']

recalls = []
candidate_counts = []

for _, row in val_source1_keyed.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts.append(len(candidate_set))
    
    if len(true_set) == 0:
        # No true matches — recall undefined here, skip from recall average
        continue
    
    found = len(true_set & candidate_set)
    recalls.append(found / len(true_set))

print("Average candidates per entity:", sum(candidate_counts) / len(candidate_counts))
print("Candidate recall (macro-average, over entities with true matches):", sum(recalls) / len(recalls))
print("Entities with true matches evaluated:", len(recalls))

Average candidates per entity: 5680.185884698605
Candidate recall (macro-average, over entities with true matches): 0.7563608887127291
Entities with true matches evaluated: 416780


### Baseline Blocking Results

The first blocking experiment used the country and the first token of the
normalized business name as the blocking key.

On the 20% validation split, the Source 2 index contained 363,912 distinct
blocking keys and the Source 3 index contained 412,262 distinct keys.

The resulting candidate sets contained an average of 5,680.19 candidates per
Source 1 entity. Candidate recall was 75.64% across validation entities with at
least one known match.

This means that approximately 24% of true matches were excluded during
blocking and therefore could not be recovered by any downstream matching
model.

The baseline demonstrates that simple blocking can reduce the search space,
but the candidate sets are still large and the recall ceiling is not high
enough for the final pipeline. The next experiment will therefore investigate
a blocking strategy that improves recall without producing unnecessarily large
candidate sets.

In [29]:
# 1. Are a few keys responsible for most of the bloat? (bucket size distribution)
import numpy as np

bucket_sizes_s2 = [len(v) for v in index_source2.values()]
bucket_sizes_s3 = [len(v) for v in index_source3.values()]

print("Source 2 bucket sizes — mean:", np.mean(bucket_sizes_s2), "median:", np.median(bucket_sizes_s2), "max:", np.max(bucket_sizes_s2))
print("Source 3 bucket sizes — mean:", np.mean(bucket_sizes_s3), "median:", np.median(bucket_sizes_s3), "max:", np.max(bucket_sizes_s3))

Source 2 bucket sizes — mean: 13.83470729187276 median: 1.0 max: 40489
Source 3 bucket sizes — mean: 12.820980347448952 median: 1.0 max: 50862


In [30]:
# Inspect the largest blocking buckets

largest_s2 = sorted(
    index_source2.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

largest_s3 = sorted(
    index_source3.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

print("Largest Source 2 buckets:")
for key, ids in largest_s2:
    print(key, "->", len(ids))

print("\nLargest Source 3 buckets:")
for key, ids in largest_s3:
    print(key, "->", len(ids))

Largest Source 2 buckets:
US||the -> 40489
US||pediatric -> 30497
India||m -> 25796
India||shri -> 25702
India||dr -> 25673
India||sri -> 25658
India||smt -> 25474
India||mr -> 25449
India||new -> 14299
US||family -> 13231
US||a -> 11974
India||india -> 10655
US||eye -> 10532
US||ear -> 10393
India||mumbai -> 10384
US||blue -> 10379
US||physical -> 10338
US||dental -> 10315
US||foot -> 10272
US||primary -> 10221

Largest Source 3 buckets:
US||the -> 50862
US||pediatric -> 30944
India||m -> 29422
India||mr -> 29388
India||sri -> 29359
India||smt -> 29274
India||shri -> 29199
India||dr -> 29189
India||new -> 15783
US||family -> 13187
US||a -> 11972
India||india -> 10909
India||shree -> 10679
US||ear -> 10652
US||primary -> 10522
US||dental -> 10514
US||eye -> 10463
US||physical -> 10371
US||foot -> 10299
US||blue -> 10235


### Blocking Bucket Analysis

The first-token blocking strategy produced highly uneven bucket sizes. Although
the median bucket size was only 1 for both Source 2 and Source 3, a small number
of blocking keys produced extremely large buckets.

The largest buckets were associated with common or generic first tokens such as
`the`, `m`, `mr`, `dr`, `smt`, `sri`, and `shri`. For example, the largest Source
2 bucket contained 40,489 records for the key `US||the`, while the largest
Source 3 bucket contained 50,862 records for the same key.

This shows that the high average candidate count is driven disproportionately
by a relatively small number of very large buckets. It also suggests that
using the first token alone does not provide sufficiently selective blocking
for records beginning with common prefixes or generic terms.

We will next examine the true matches that were missed by this blocking
strategy before deciding how to modify the blocking key.

In [31]:
# 2. For entities where we LOST recall (missed a true match), was the true match's first word actually different?
# Sample a handful of failure cases to eyeball
sample_checked = 0
for _, row in val_source1_keyed.iterrows():
    entity_id = row['entity_id']
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    if len(true_set) == 0:
        continue
    candidate_set = get_candidates(row['block_key'])
    missed = true_set - candidate_set
    if missed:
        print("Source1:", row['business_name'], "| norm:", row['norm_name'])
        # look up one missed match's actual name
        missed_id = next(iter(missed))
        if missed_id.startswith('S2-'):
            match_row = train_source2[train_source2['entity_id'] == missed_id]
        else:
            match_row = train_source3[train_source3['entity_id'] == missed_id]
        print("  Missed match:", match_row['business_name'].values)
        sample_checked += 1
        if sample_checked >= 8:
            break

Source1: Prime Money | norm: prime money
  Missed match: ['primemoney.com']
Source1: Dream Construction Limited | norm: dream construction
  Missed match: ['డ్రీమ్ కన్\u200cస్ట్రక్షన్ లిమిటెడ్']
Source1: Rajani Enterprises of Kannur | norm: rajani enterprises of kannur
  Missed match: ['M/s Rajani Enterprises of']
Source1: Schaefer, Michael and Silver Associates | norm: schaefer michael and silver associates
  Missed match: ['Silver Michael and Schaefer, Ass0ciates']
Source1: Aggie E. Nagle, L.C.S.W. | norm: aggie e nagle l c s w
  Missed match: ['aggieenagle.com']
Source1: Pacific Learning Laboratories LLC | norm: pacific learning laboratories
  Missed match: ['... The Pacific Learning Laboratories [LLC] [LLC]']
Source1: Mayur Hospital of Gondal | norm: mayur hospital of gondal
  Missed match: ['M/s Mayur Hospital of Gondal Corporation']
Source1: Orchid Renewable Ltd | norm: orchid renewable
  Missed match: ['Smt BRIXUMBRABELO']


### Baseline Failure Analysis

The missed-match examples show that the first-token blocking strategy fails for
several different reasons. Some records contain leading noise prefixes such as
`M/s`, while others use alternative representations such as domain names or
different scripts. These cases indicate that improving blocking will require
more than simply changing the first-token rule.

We first address the clearly observed prefix and honorific noise because these
tokens also contributed to the largest blocking buckets. Cross-script and
other name-variation cases will be investigated separately in subsequent
experiments.

### 4.5 Experiment — Noise-Prefix Normalization

In [32]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )
    
    stopwords = {
        'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
        'corp', 'corporation', 'llc', 'llp', 'co', 'company',
        'dr', 'mr', 'mrs', 'ms', 'smt', 'sri', 'shri', 'prof',
        # leading noise prefixes seen in real data
        'm', 's',  # "m/s" becomes "m s" after punctuation stripping
    }
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [33]:
train_source2_keyed_v2 = build_block_key(train_source2)
train_source3_keyed_v2 = build_block_key(train_source3)
val_source1_keyed_v2 = build_block_key(val_source1)

print("Done building v2 keys.")

Done building v2 keys.


In [34]:
index_source2_v2 = build_index(train_source2_keyed_v2)
index_source3_v2 = build_index(train_source3_keyed_v2)

print("Number of distinct keys in Source 2 index (v2):", len(index_source2_v2))
print("Number of distinct keys in Source 3 index (v2):", len(index_source3_v2))

Number of distinct keys in Source 2 index (v2): 372168
Number of distinct keys in Source 3 index (v2): 420329


In [35]:
bucket_sizes_s2_v2 = [len(v) for v in index_source2_v2.values()]
bucket_sizes_s3_v2 = [len(v) for v in index_source3_v2.values()]

print("Source 2 (v2) bucket sizes — mean:", np.mean(bucket_sizes_s2_v2), "median:", np.median(bucket_sizes_s2_v2), "max:", np.max(bucket_sizes_s2_v2))
print("Source 3 (v2) bucket sizes — mean:", np.mean(bucket_sizes_s3_v2), "median:", np.median(bucket_sizes_s3_v2), "max:", np.max(bucket_sizes_s3_v2))

Source 2 (v2) bucket sizes — mean: 13.527804647363556 median: 1.0 max: 40489
Source 3 (v2) bucket sizes — mean: 12.574918694641578 median: 1.0 max: 50863


In [36]:
def get_candidates_v2(block_key):
    c2 = index_source2_v2.get(block_key, [])
    c3 = index_source3_v2.get(block_key, [])
    return set(c2) | set(c3)

recalls_v2 = []
candidate_counts_v2 = []

for _, row in val_source1_keyed_v2.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates_v2(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts_v2.append(len(candidate_set))
    
    if len(true_set) == 0:
        continue
    
    found = len(true_set & candidate_set)
    recalls_v2.append(found / len(true_set))

print("Average candidates per entity (v2):", sum(candidate_counts_v2) / len(candidate_counts_v2))
print("Candidate recall (v2):", sum(recalls_v2) / len(recalls_v2))
print("Entities with true matches evaluated:", len(recalls_v2))

Average candidates per entity (v2): 5846.941461148936
Candidate recall (v2): 0.784331800797802
Entities with true matches evaluated: 416780


## Experiment 4.5 Results — Noise-Prefix Normalization

The second normalization experiment removed common honorifics and leading noise prefixes observed in the training data, including `dr`, `mr`, `mrs`, `ms`, `smt`, `sri`, `shri`, `prof`, and the `m/s` representation after punctuation normalization.

This modification increased candidate recall from 75.64% to 78.43%, an improvement of approximately 2.80 percentage points.

The average candidate count increased slightly from 5,680.19 to 5,846.94 candidates per Source 1 entity. The maximum bucket sizes remained almost unchanged: the largest Source 2 bucket contained 40,489 records in both versions, while the largest Source 3 bucket changed from 50,862 to 50,863 records.

This indicates that removing noise prefixes improved the ability to retrieve true matches, but did not substantially reduce the largest blocking buckets. The increase in average candidate count suggests that normalization caused some records to be grouped into the same blocking keys.

Overall, this normalization change improved candidate recall and will therefore be retained for subsequent experiments.

## 4.6 Experiment — Generic Token Normalization

The previous experiment removed common honorifics and noise prefixes such as `dr`, `mr`, `smt`, `sri`, `shri`, and `m/s`. However, the blocking analysis also showed that the token `the` produced very large buckets, particularly in Source 2 and Source 3.

In this experiment, `the` was added to the normalization stopword list. The goal was to determine whether removing this common generic token could improve blocking selectivity without losing true matches.

In [37]:
def normalize_name(name):
    if pd.isna(name):
        return ""
    
    name = name.lower()
    name = ''.join(
        ' ' if unicodedata.category(ch)[0] in ('P', 'S') else ch
        for ch in name
    )
    
    stopwords = {
        'inc', 'incorporated', 'ltd', 'limited', 'pvt', 'private',
        'corp', 'corporation', 'llc', 'llp', 'co', 'company',
        'dr', 'mr', 'mrs', 'ms', 'smt', 'sri', 'shri', 'prof',
        'm', 's',
        'the'
    }
    tokens = [t for t in name.split() if t not in stopwords]
    
    return ' '.join(tokens).strip()

In [38]:
train_source2_keyed_v3 = build_block_key(train_source2)
train_source3_keyed_v3 = build_block_key(train_source3)
val_source1_keyed_v3 = build_block_key(val_source1)

In [39]:
index_source2_v3 = build_index(train_source2_keyed_v3)
index_source3_v3 = build_index(train_source3_keyed_v3)

print("Number of distinct keys in Source 2 index (v3):", len(index_source2_v3))
print("Number of distinct keys in Source 3 index (v3):", len(index_source3_v3))

Number of distinct keys in Source 2 index (v3): 372673
Number of distinct keys in Source 3 index (v3): 420923


In [40]:
bucket_sizes_s2_v3 = [len(v) for v in index_source2_v3.values()]
bucket_sizes_s3_v3 = [len(v) for v in index_source3_v3.values()]

print("Source 2 (v3) bucket sizes — mean:", np.mean(bucket_sizes_s2_v3), "median:", np.median(bucket_sizes_s2_v3), "max:", np.max(bucket_sizes_s2_v3))
print("Source 3 (v3) bucket sizes — mean:", np.mean(bucket_sizes_s3_v3), "median:", np.median(bucket_sizes_s3_v3), "max:", np.max(bucket_sizes_s3_v3))

Source 2 (v3) bucket sizes — mean: 13.509473452597854 median: 1.0 max: 30875
Source 3 (v3) bucket sizes — mean: 12.557173164688079 median: 1.0 max: 31445


In [41]:
def get_candidates_v3(block_key):
    c2 = index_source2_v3.get(block_key, [])
    c3 = index_source3_v3.get(block_key, [])
    return set(c2) | set(c3)

recalls_v3 = []
candidate_counts_v3 = []

for _, row in val_source1_keyed_v3.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates_v3(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts_v3.append(len(candidate_set))
    
    if len(true_set) == 0:
        continue
    
    found = len(true_set & candidate_set)
    recalls_v3.append(found / len(true_set))

print("Average candidates per entity (v3):", sum(candidate_counts_v3) / len(candidate_counts_v3))
print("Candidate recall (v3):", sum(recalls_v3) / len(recalls_v3))
print("Entities with true matches evaluated:", len(recalls_v3))

Average candidates per entity (v3): 5830.759423606312
Candidate recall (v3): 0.7926473105808727
Entities with true matches evaluated: 416780


## Experiment 4.6 Results — Generic Token Normalization

Removing the generic token `the` improved candidate recall from 78.43% to 79.26%, an improvement of approximately 0.83 percentage points.

At the same time, the average number of candidates per Source 1 entity decreased slightly from 5,846.94 to 5,830.76. This means the modification improved recall while also slightly reducing the overall candidate volume.

The maximum blocking bucket sizes were also substantially smaller than the original baseline, with the largest Source 2 bucket containing 30,875 records and the largest Source 3 bucket containing 31,445 records.

Overall, removing `the` produced a small but positive improvement in both candidate recall and candidate efficiency. This version will therefore be retained for the matching stage.

## 5. Matching Stage — Computational Feasibility Test

Before running similarity scoring over the full validation set, we first measure the actual runtime on a small sample of 50 validation entities.

The blocking stage has already produced the candidate pairs. The next step is to calculate name and address similarity for each candidate pair. Because the dataset contains millions of records, this timing test helps estimate the computational cost of scoring the full validation set before committing to a long-running computation.

This is a feasibility test rather than a leaderboard experiment. The results will be used to decide how the matching stage should be implemented efficiently.

In [42]:
from rapidfuzz import fuzz

def pair_similarity(name1, addr1, name2, addr2):
    """Returns a combined similarity score (0-100) for one candidate pair."""
    name_sim = fuzz.token_sort_ratio(str(name1), str(name2)) if pd.notna(name1) and pd.notna(name2) else 0
    
    if pd.isna(addr1) or pd.isna(addr2):
        addr_sim = 0
    else:
        addr_sim = fuzz.token_sort_ratio(str(addr1), str(addr2))
    
    return 0.6 * name_sim + 0.4 * addr_sim

In [43]:
source2_lookup = train_source2_keyed_v3.set_index('entity_id')[['business_name', 'business_address']].to_dict('index')
source3_lookup = train_source3_keyed_v3.set_index('entity_id')[['business_name', 'business_address']].to_dict('index')

print("Lookups built.")

Lookups built.


In [44]:
import time

sample_val = val_source1_keyed_v3.head(50)

start = time.time()

sample_results = []
for _, row in sample_val.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    name1 = row['business_name']
    addr1 = row['business_address']
    
    candidate_ids = list(get_candidates_v3(block_key))
    
    for cand_id in candidate_ids:
        lookup = source2_lookup if cand_id.startswith('S2-') else source3_lookup
        cand_data = lookup.get(cand_id)
        if cand_data is None:
            continue
        
        name2 = cand_data['business_name']
        addr2 = cand_data['business_address']
        
        score = pair_similarity(name1, addr1, name2, addr2)
        sample_results.append((entity_id, cand_id, score))

elapsed = time.time() - start
print(f"Time for 50 entities: {elapsed:.2f} seconds")
print(f"Estimated total pairs scored: {len(sample_results)}")
print(f"Estimated time for all {len(val_source1_keyed_v3)} entities: {elapsed / 50 * len(val_source1_keyed_v3) / 60:.1f} minutes")

Time for 50 entities: 3.54 seconds
Estimated total pairs scored: 337398
Estimated time for all 441365 entities: 520.7 minutes


### 5.3 Batched RapidFuzz Scoring — Timing Test

The previous implementation improved candidate lookup by using dictionary-based lookups, but scoring all candidate pairs individually was still estimated to require several hours.

This experiment tests batched RapidFuzz scoring using `process.cdist()`. Instead of calling the similarity function separately for every candidate pair, candidate names and addresses are collected into arrays and scored in batches.

The purpose is to determine whether batched similarity computation can substantially reduce the runtime while producing the same type of similarity scores.

In [45]:
from rapidfuzz import process, fuzz
import numpy as np

def score_entity_candidates(name1, addr1, candidate_ids):
    """Score one Source 1 entity against all its candidates, batched."""
    cand_names = []
    cand_addrs = []
    valid_ids = []
    
    for cand_id in candidate_ids:
        lookup = source2_lookup if cand_id.startswith('S2-') else source3_lookup
        cand_data = lookup.get(cand_id)
        if cand_data is None:
            continue
        valid_ids.append(cand_id)
        cand_names.append(str(cand_data['business_name']) if pd.notna(cand_data['business_name']) else "")
        cand_addrs.append(str(cand_data['business_address']) if pd.notna(cand_data['business_address']) else "")
    
    if not valid_ids:
        return []
    
    name1_str = str(name1) if pd.notna(name1) else ""
    addr1_str = str(addr1) if pd.notna(addr1) else ""
    
    name_scores = process.cdist([name1_str], cand_names, scorer=fuzz.token_sort_ratio)[0]
    addr_scores = process.cdist([addr1_str], cand_addrs, scorer=fuzz.token_sort_ratio)[0]
    
    combined = 0.6 * name_scores + 0.4 * addr_scores
    return list(zip(valid_ids, combined))

In [46]:
import time

sample_val = val_source1_keyed_v3.head(50)

start = time.time()

sample_results = []
for _, row in sample_val.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    name1 = row['business_name']
    addr1 = row['business_address']
    
    candidate_ids = list(get_candidates_v3(block_key))
    scored = score_entity_candidates(name1, addr1, candidate_ids)
    
    for cand_id, score in scored:
        sample_results.append((entity_id, cand_id, score))

elapsed = time.time() - start
print(f"Time for 50 entities: {elapsed:.2f} seconds")
print(f"Estimated total pairs scored: {len(sample_results)}")
print(f"Estimated time for all {len(val_source1_keyed_v3)} entities: {elapsed / 50 * len(val_source1_keyed_v3) / 60:.1f} minutes")

Time for 50 entities: 2.64 seconds
Estimated total pairs scored: 337398
Estimated time for all 441365 entities: 388.9 minutes


### 5.4 Token-Overlap Pre-filter

The previous timing experiments showed that even with optimized dictionary lookups and batched RapidFuzz scoring, scoring every candidate produced by V3 blocking would take several hours.

To reduce the number of pairs reaching the more expensive fuzzy-matching stage, this experiment introduces a cheap token-overlap pre-filter.

For each Source 1 entity, the normalized business name is compared with the normalized names of its V3 candidates using Jaccard similarity. Candidates are ranked by their token overlap, and only the top K candidates are retained for subsequent RapidFuzz scoring.

The purpose of this experiment is to measure whether this inexpensive filtering step can substantially reduce the candidate set while retaining most of the true matches already recovered by V3 blocking.

Before applying the filter to the full validation set, its recall will be measured on a smaller validation sample.

#### Step 1 — Define the Jaccard Pre-filter

The pre-filter calculates Jaccard similarity between the sets of normalized name tokens of the Source 1 entity and each candidate. Candidates with the highest token overlap are retained.

In [47]:
def jaccard_prefilter(name1, candidate_ids, keyed_lookup_dict, top_k=50):
    """
    name1: normalized Source 1 name (string)
    candidate_ids: list of candidate entity_ids
    keyed_lookup_dict: entity_id -> {'norm_name': ..., 'business_address': ..., 'business_name': ...}
    Returns top_k candidate_ids ranked by Jaccard token overlap.
    """
    tokens1 = set(name1.split())
    
    scored = []
    for cand_id in candidate_ids:
        cand_data = keyed_lookup_dict.get(cand_id)
        if cand_data is None:
            continue
        tokens2 = set(cand_data['norm_name'].split())
        
        union_size = len(tokens1 | tokens2)
        if union_size == 0:
            jaccard = 0
        else:
            jaccard = len(tokens1 & tokens2) / union_size
        
        scored.append((cand_id, jaccard))
    
    scored.sort(key=lambda x: x[1], reverse=True)
    return [cand_id for cand_id, _ in scored[:top_k]]

#### Step 2 — Build Candidate Lookups

The existing candidate lookups contain the original business name and address. For the Jaccard pre-filter, the normalized name is also required, so new lookup dictionaries containing `norm_name` are created.

In [48]:
source2_lookup_full = train_source2_keyed_v3.set_index('entity_id')[['business_name', 'business_address', 'norm_name']].to_dict('index')
source3_lookup_full = train_source3_keyed_v3.set_index('entity_id')[['business_name', 'business_address', 'norm_name']].to_dict('index')

print("Full lookups built.")

Full lookups built.


#### Step 3 — Measure Pre-filter Recall

Before applying the pre-filter to the full validation set, it is evaluated on a sample of 500 validation entities.

For each entity with known true matches, we check how many of its true matches remain within the top-50 candidates selected by the Jaccard pre-filter.

This measures whether the pre-filter is safe enough to place before RapidFuzz scoring. A large loss in recall would indicate that the top-50 cutoff is too aggressive and should be increased or replaced.

In [49]:
sample_val = val_source1_keyed_v3.head(500)  # bigger sample since this step is cheap

prefilter_recalls = []

for _, row in sample_val.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    norm_name1 = row['norm_name']
    
    candidate_ids = list(get_candidates_v3(block_key))
    
    def lookup_fn(cid):
        return source2_lookup_full.get(cid) if cid.startswith('S2-') else source3_lookup_full.get(cid)
    
    combined_lookup = {cid: lookup_fn(cid) for cid in candidate_ids}
    
    filtered_ids = jaccard_prefilter(norm_name1, candidate_ids, combined_lookup, top_k=50)
    
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    if len(true_set) == 0:
        continue
    
    found = len(true_set & set(filtered_ids))
    prefilter_recalls.append(found / len(true_set))

print("Pre-filter recall (top-50, sample of 500):", sum(prefilter_recalls) / len(prefilter_recalls))
print("Entities with true matches in sample:", len(prefilter_recalls))

Pre-filter recall (top-50, sample of 500): 0.62677384344051
Entities with true matches in sample: 468


### Initial Result

The top-50 Jaccard pre-filter retained 62.79% of the true matches on a sample of 500 validation entities with true matches.

This indicates that a top-50 cutoff is too aggressive for the current blocking strategy, so it will not be used as the final pre-filter cutoff. Additional K values will be evaluated to determine whether a larger candidate set can provide a better recall-efficiency trade-off.

#### Testing Multiple Top-K Cutoffs

The initial top-50 pre-filter retained only 62.79% of true matches in the 500-entity sample, indicating that the cutoff was too aggressive.

To find a better balance between computational efficiency and recall, the same validation sample is now evaluated using several larger candidate cutoffs: 50, 100, 200, 300, and 500.

The goal is to identify how recall changes as more candidates are retained before RapidFuzz scoring.

In [50]:
for k in [50, 100, 200, 300, 500]:
    prefilter_recalls = []
    
    for _, row in sample_val.iterrows():
        entity_id = row['entity_id']
        block_key = row['block_key']
        norm_name1 = row['norm_name']
        
        candidate_ids = list(get_candidates_v3(block_key))
        
        def lookup_fn(cid):
            return source2_lookup_full.get(cid) if cid.startswith('S2-') else source3_lookup_full.get(cid)
        
        combined_lookup = {cid: lookup_fn(cid) for cid in candidate_ids}
        
        filtered_ids = jaccard_prefilter(norm_name1, candidate_ids, combined_lookup, top_k=k)
        
        true_set = parse_ids(val_gt_lookup.get(entity_id))
        if len(true_set) == 0:
            continue
        
        found = len(true_set & set(filtered_ids))
        prefilter_recalls.append(found / len(true_set))
    
    print(f"K={k}: recall = {sum(prefilter_recalls) / len(prefilter_recalls):.4f}")
    

K=50: recall = 0.6268
K=100: recall = 0.6742
K=200: recall = 0.7141
K=300: recall = 0.7241
K=500: recall = 0.7344


In [51]:
import os
print("CPU cores available:", os.cpu_count())


CPU cores available: 4


### 4.7 Experiment — Address-Augmented Blocking

The V3 blocking strategy uses country and the first normalized business-name token as the blocking key. Although V3 improved candidate recall to 79.26%, the resulting candidate set remains large, averaging approximately 5,831 candidates per Source 1 entity.

A new blocking strategy is therefore being investigated that adds a coarse address signal to the existing name-based key. The initial hypothesis is that combining the first normalized name token with an address token may split large blocking buckets into smaller groups and reduce the number of candidate pairs.

Before rebuilding the full Source 2 and Source 3 indexes, the proposed address token is inspected on a small sample to determine whether it provides a useful locality signal or is dominated by house numbers and other non-informative values.

In [52]:
def extract_city_token(address):
    """Very cheap: take the first alphabetic token from the address as a coarse locality signal."""
    if pd.isna(address):
        return ""
    addr_norm = normalize_name(address)  # reuse existing normalization
    tokens = addr_norm.split()
    return tokens[0] if tokens else ""

def build_block_key_v4(df):
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['addr_token'] = df['business_address'].apply(extract_city_token)
    df['block_key'] = df['country'] + '||' + df['norm_name'].apply(first_token) + '||' + df['addr_token']
    return df

In [53]:
sample_check = train_source1.head(15).copy()
sample_check['norm_name'] = sample_check['business_name'].apply(normalize_name)
sample_check['addr_token'] = sample_check['business_address'].apply(extract_city_token)
print(sample_check[['business_name', 'business_address', 'norm_name', 'addr_token']].to_string())

                               business_name                                                                                                           business_address                   norm_name addr_token
0                        Orelee's Barbershop                                                                                     1795 Westchester Drive, High Point, NC           orelee barbershop       1795
1                                Prime Money                                                                                            17560 Ellis Road, Tahlequah, OK                 prime money      17560
2                              B+ Retail Inc                                                                                        1712 Montebello Avenue, Phoenix, AZ                    b retail       1712
3                              Christ Chapel                                                                          2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD     

#### Initial Inspection

The sample shows that the first normalized address token is frequently a house number, unit indicator, or other structural component rather than a reliable locality signal. Examples include `1795`, `17560`, `1712`, `797`, `2505`, `1111`, `unit`, and `h`.

Because this token does not consistently represent a useful geographic or address component, it is not suitable as the second blocking key in its current form. The full V4 indexes will therefore not be built using this feature.

#### V4 Candidate — Last Address Segment

The previous address-based blocking attempt used the first address token, but inspection showed that it was frequently a house number, unit indicator, or other structural component rather than a useful locality signal.

As an alternative, this experiment investigates the last comma-separated segment of the address. Business addresses often place geographic information such as a city, state, or region toward the end of the address.

The last segment is therefore tested as a coarse address signal. This is only an initial inspection; the full blocking indexes will not be rebuilt unless the sample suggests that the signal is useful.

In [54]:
def extract_last_segment(address):
    """Take the last comma-separated segment of the address as a coarse locality signal."""
    if pd.isna(address):
        return ""
    segments = address.split(',')
    last = segments[-1].strip()
    return normalize_name(last)  # reuse existing normalization for consistency

sample_check = train_source1.head(15).copy()
sample_check['norm_name'] = sample_check['business_name'].apply(normalize_name)
sample_check['addr_last_seg'] = sample_check['business_address'].apply(extract_last_segment)
print(sample_check[['business_name', 'business_address', 'norm_name', 'addr_last_seg']].to_string())

                               business_name                                                                                                           business_address                   norm_name        addr_last_seg
0                        Orelee's Barbershop                                                                                     1795 Westchester Drive, High Point, NC           orelee barbershop                   nc
1                                Prime Money                                                                                            17560 Ellis Road, Tahlequah, OK                 prime money                   ok
2                              B+ Retail Inc                                                                                        1712 Montebello Avenue, Phoenix, AZ                    b retail                   az
3                              Christ Chapel                                                                          2100 Cameron D

#### V4 Address Signal Inspection — Result

The sample shows that the last comma-separated address segment frequently contains geographic information such as a state, province, or region. Examples include `nc`, `ok`, `az`, `west bengal`, `maharashtra`, `tn`, `in`, and `telangana`.

Some addresses use a different ordering and therefore produce a street or locality value instead. However, the signal appears substantially more informative than the first address token, which was frequently a house number or unit indicator.

The last address segment will therefore be tested as an additional blocking signal. The full indexes will be evaluated for candidate recall and candidate volume before deciding whether to retain this strategy.

#### V4 — Address-Augmented Blocking: Bucket Size Feasibility Test

The V3 blocking strategy uses country and the first normalized business-name token. To reduce the large candidate volume, this experiment adds the last comma-separated address segment as a second blocking signal.

Before rebuilding the full Source 2 and Source 3 indexes, the new blocking key is tested on a 200,000-record Source 2 sample. The purpose is to determine whether the additional address signal substantially reduces bucket sizes while remaining computationally practical.

If the resulting buckets are sufficiently smaller, the strategy will then be evaluated on the validation set for candidate recall before being considered for the final pipeline.

In [55]:
def build_block_key_v5(df):
    df = df.copy()
    df['norm_name'] = df['business_name'].apply(normalize_name)
    df['addr_last_seg'] = df['business_address'].apply(extract_last_segment)
    df['block_key'] = df['country'] + '||' + df['norm_name'].apply(first_token) + '||' + df['addr_last_seg']
    return df

In [56]:
sample_s2 = train_source2.sample(n=200000, random_state=42)
sample_s2_keyed = build_block_key_v5(sample_s2)

bucket_sizes_sample = sample_s2_keyed.groupby('block_key').size()
print("Sample bucket sizes — mean:", bucket_sizes_sample.mean(), "median:", bucket_sizes_sample.median(), "max:", bucket_sizes_sample.max())
print("Number of distinct keys:", len(bucket_sizes_sample))

Sample bucket sizes — mean: 1.6035534744994908 median: 1.0 max: 292
Number of distinct keys: 124723


In [57]:
build_block_key_v4 = build_block_key_v5  # alias for consistent naming going forward

In [58]:
train_source2_keyed_v4 = build_block_key_v5(train_source2)
train_source3_keyed_v4 = build_block_key_v5(train_source3)
val_source1_keyed_v4 = build_block_key_v5(val_source1)

print("Done building v4 keys.")

Done building v4 keys.


In [59]:
index_source2_v4 = build_index(train_source2_keyed_v4)
index_source3_v4 = build_index(train_source3_keyed_v4)

print("Number of distinct keys in Source 2 index (v4):", len(index_source2_v4))
print("Number of distinct keys in Source 3 index (v4):", len(index_source3_v4))

Number of distinct keys in Source 2 index (v4): 1540612
Number of distinct keys in Source 3 index (v4): 1799828


In [60]:
bucket_sizes_s2_v4 = [len(v) for v in index_source2_v4.values()]
bucket_sizes_s3_v4 = [len(v) for v in index_source3_v4.values()]

print(
    "Source 2 (v4) bucket sizes — mean:",
    np.mean(bucket_sizes_s2_v4),
    "median:",
    np.median(bucket_sizes_s2_v4),
    "max:",
    np.max(bucket_sizes_s2_v4)
)

print(
    "Source 3 (v4) bucket sizes — mean:",
    np.mean(bucket_sizes_s3_v4),
    "median:",
    np.median(bucket_sizes_s3_v4),
    "max:",
    np.max(bucket_sizes_s3_v4)
)

Source 2 (v4) bucket sizes — mean: 3.2679324839739015 median: 1.0 max: 7118
Source 3 (v4) bucket sizes — mean: 2.93672673166547 median: 1.0 max: 6651


In [61]:
def get_candidates_v4(block_key):
    c2 = index_source2_v4.get(block_key, [])
    c3 = index_source3_v4.get(block_key, [])
    return set(c2) | set(c3)

recalls_v4 = []
candidate_counts_v4 = []

for _, row in val_source1_keyed_v4.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    
    candidate_set = get_candidates_v4(block_key)
    true_set = parse_ids(val_gt_lookup.get(entity_id))
    
    candidate_counts_v4.append(len(candidate_set))
    
    if len(true_set) == 0:
        continue
    
    found = len(true_set & candidate_set)
    recalls_v4.append(found / len(true_set))

print(
    "Average candidates per entity (v4):",
    sum(candidate_counts_v4) / len(candidate_counts_v4)
)

print(
    "Candidate recall (v4):",
    sum(recalls_v4) / len(recalls_v4)
)

print(
    "Entities with true matches evaluated:",
    len(recalls_v4)
)

Average candidates per entity (v4): 163.58522084895722
Candidate recall (v4): 0.2558794237404655
Entities with true matches evaluated: 416780


## Experiment 4.7 Results — Address-Augmented Blocking

Adding the last comma-separated address segment to the V3 blocking key reduced the average candidate count dramatically, from approximately 5,831 to 164 candidates per Source 1 entity.

However, this reduction came at a major cost to candidate recall. Candidate recall decreased from 79.26% in V3 to only 25.59% in V4.

This indicates that the additional address constraint is too restrictive for the noisy entity-resolution data. Many true matches do not share the same final address segment even when they refer to the same business.

Since a true match excluded during blocking cannot be recovered by the later matching model, preserving candidate recall is critical. Therefore, V4 will not be used for the final matching pipeline.

V3 remains the current blocking strategy because it provides substantially higher candidate recall while avoiding the overly restrictive address constraint.

### 5.1 Experiment — Sampled Validation of RapidFuzz Matching on V3 Candidates

Blocking experiments are frozen for now. V3 (first-significant-word + country blocking,
with honorific/prefix stopword cleanup) is our accepted baseline: ~79.26% candidate
recall at ~5,831 candidates per Source 1 entity.

V4 (adding last-address-segment to the blocking key) was tested and rejected: it cut
average candidates to ~164 but collapsed candidate recall to ~25.59%, because address
field order and completeness vary too much across sources to use as a hard blocking
constraint.

Running RapidFuzz matching over the full 441K-entity validation set at V3's candidate
volume is estimated at ~10 hours, which isn't practical under the current time
constraint. This experiment instead scores a random sample of 20,000 Source 1
validation entities (random_state=42), using the same V3 blocking, batched RapidFuzz
`token_sort_ratio` scoring (0.6 name / 0.4 address), and computes precision, recall,
and F_0.5 at several similarity thresholds — scoring each candidate pair once and
sweeping thresholds over the stored scores, rather than rescoring per threshold.

Goal: pick a reasonable threshold to generate a first submittable `matching_results.tsv`
and `candidate_pairs.tsv`, not to finalize the matching approach.

In [62]:
sample_val_20k = val_source1_keyed_v3.sample(n=20000, random_state=42).reset_index(drop=True)
print("Sampled entities:", len(sample_val_20k))

Sampled entities: 20000


In [63]:
""" import time

start = time.time()

entity_scores = {}  # entity_id -> (true_set, [(cand_id, score), ...])

for i, row in sample_val_20k.iterrows():
    entity_id = row['entity_id']
    block_key = row['block_key']
    name1 = row['business_name']
    addr1 = row['business_address']

    candidate_ids = list(get_candidates_v3(block_key))
    scored = score_entity_candidates(name1, addr1, candidate_ids)  # reuses existing batched function

    true_set = parse_ids(val_gt_lookup.get(entity_id))
    entity_scores[entity_id] = (true_set, scored)

    if (i + 1) % 2000 == 0:
        elapsed = time.time() - start
        print(f"Processed {i+1}/20000 entities, {elapsed:.1f}s elapsed")

elapsed = time.time() - start
print(f"Done. Total time: {elapsed/60:.1f} minutes") """

' import time\n\nstart = time.time()\n\nentity_scores = {}  # entity_id -> (true_set, [(cand_id, score), ...])\n\nfor i, row in sample_val_20k.iterrows():\n    entity_id = row[\'entity_id\']\n    block_key = row[\'block_key\']\n    name1 = row[\'business_name\']\n    addr1 = row[\'business_address\']\n\n    candidate_ids = list(get_candidates_v3(block_key))\n    scored = score_entity_candidates(name1, addr1, candidate_ids)  # reuses existing batched function\n\n    true_set = parse_ids(val_gt_lookup.get(entity_id))\n    entity_scores[entity_id] = (true_set, scored)\n\n    if (i + 1) % 2000 == 0:\n        elapsed = time.time() - start\n        print(f"Processed {i+1}/20000 entities, {elapsed:.1f}s elapsed")\n\nelapsed = time.time() - start\nprint(f"Done. Total time: {elapsed/60:.1f} minutes") '

In [64]:
"""def evaluate_at_threshold(entity_scores, threshold):
    scores = []
    for entity_id, (true_set, scored) in entity_scores.items():
        pred_set = {cand_id for cand_id, score in scored if score >= threshold}
        scores.append(f_beta_score(true_set, pred_set))
    return sum(scores) / len(scores)

def precision_recall_at_threshold(entity_scores, threshold):
    total_tp, total_pred, total_true = 0, 0, 0
    for entity_id, (true_set, scored) in entity_scores.items():
        pred_set = {cand_id for cand_id, score in scored if score >= threshold}
        total_tp += len(true_set & pred_set)
        total_pred += len(pred_set)
        total_true += len(true_set)
    precision = total_tp / total_pred if total_pred > 0 else 0
    recall = total_tp / total_true if total_true > 0 else 0
    return precision, recall

thresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]

print(f"{'Threshold':>10} {'F0.5':>8} {'Precision':>10} {'Recall':>8}")
for t in thresholds:
    f05 = evaluate_at_threshold(entity_scores, t)
    prec, rec = precision_recall_at_threshold(entity_scores, t)
    print(f"{t:>10} {f05:>8.4f} {prec:>10.4f} {rec:>8.4f}")"""

'def evaluate_at_threshold(entity_scores, threshold):\n    scores = []\n    for entity_id, (true_set, scored) in entity_scores.items():\n        pred_set = {cand_id for cand_id, score in scored if score >= threshold}\n        scores.append(f_beta_score(true_set, pred_set))\n    return sum(scores) / len(scores)\n\ndef precision_recall_at_threshold(entity_scores, threshold):\n    total_tp, total_pred, total_true = 0, 0, 0\n    for entity_id, (true_set, scored) in entity_scores.items():\n        pred_set = {cand_id for cand_id, score in scored if score >= threshold}\n        total_tp += len(true_set & pred_set)\n        total_pred += len(pred_set)\n        total_true += len(true_set)\n    precision = total_tp / total_pred if total_pred > 0 else 0\n    recall = total_tp / total_true if total_true > 0 else 0\n    return precision, recall\n\nthresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]\n\nprint(f"{\'Threshold\':>10} {\'F0.5\':>8} {\'Precision\':>10} {\'Recall\':>8}")\nfor t in threshold

### 5.2 Optimized Threshold Sweep

The initial threshold-sweep implementation repeatedly scanned the stored
candidate scores for every threshold. Although the RapidFuzz scoring itself
completed successfully, this evaluation method caused excessive memory
pressure and very slow execution on the available hardware.

The stored `entity_scores` are retained and reused; RapidFuzz scoring is
not repeated.

This experiment uses a single pass over each entity's candidate scores.
Threshold counts are accumulated using a small difference-array approach,
allowing F_0.5, precision, and recall to be calculated for all thresholds
without repeatedly scanning the candidate list or creating large temporary
structures.

The thresholds evaluated are 50, 55, 60, 65, 70, 75, 80, 85, and 90.

In [65]:
"""import time
from bisect import bisect_right
from itertools import accumulate

thresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]
n_thresh = len(thresholds)

f05_sums = [0.0] * n_thresh
tp_totals = [0] * n_thresh
pred_totals = [0] * n_thresh
n_entities = 0
true_total = 0

start = time.time()

for entity_id, (true_set, scored) in entity_scores.items():
    n_entities += 1
    true_count = len(true_set)
    true_total += true_count

    pred_diff = [0] * (n_thresh + 1)
    tp_diff = [0] * (n_thresh + 1)

    for cand_id, score in scored:              # each candidate visited exactly once
        k = bisect_right(thresholds, score + 1)  # number of thresholds this score passes (score >= threshold)
        if k == 0:
            continue
        pred_diff[0] += 1
        pred_diff[k] -= 1
        if cand_id in true_set:
            tp_diff[0] += 1
            tp_diff[k] -= 1

    pred_counts = list(accumulate(pred_diff))[:n_thresh]
    tp_counts = list(accumulate(tp_diff))[:n_thresh]

    for j in range(n_thresh):
        pred_count = pred_counts[j]
        tp = tp_counts[j]

        if true_count == 0:
            f05 = 1.0 if pred_count == 0 else 0.0
        elif pred_count == 0:
            f05 = 0.0
        else:
            precision = tp / pred_count
            recall = tp / true_count
            f05 = 0.0 if (precision == 0 and recall == 0) else (1.25 * precision * recall) / (0.25 * precision + recall)

        f05_sums[j] += f05
        tp_totals[j] += tp
        pred_totals[j] += pred_count

    if n_entities % 5000 == 0:
        print(f"Processed {n_entities}/{len(entity_scores)} entities, {time.time()-start:.1f}s elapsed")

elapsed = time.time() - start
print(f"Done. Total time: {elapsed:.1f}s")

print(f"\n{'Threshold':>10} {'F0.5':>8} {'Precision':>10} {'Recall':>8}")
for j, t in enumerate(thresholds):
    f05_macro = f05_sums[j] / n_entities
    prec = tp_totals[j] / pred_totals[j] if pred_totals[j] > 0 else 0
    rec = tp_totals[j] / true_total if true_total > 0 else 0
    print(f"{t:>10} {f05_macro:>8.4f} {prec:>10.4f} {rec:>8.4f}")"""

'import time\nfrom bisect import bisect_right\nfrom itertools import accumulate\n\nthresholds = [50, 55, 60, 65, 70, 75, 80, 85, 90]\nn_thresh = len(thresholds)\n\nf05_sums = [0.0] * n_thresh\ntp_totals = [0] * n_thresh\npred_totals = [0] * n_thresh\nn_entities = 0\ntrue_total = 0\n\nstart = time.time()\n\nfor entity_id, (true_set, scored) in entity_scores.items():\n    n_entities += 1\n    true_count = len(true_set)\n    true_total += true_count\n\n    pred_diff = [0] * (n_thresh + 1)\n    tp_diff = [0] * (n_thresh + 1)\n\n    for cand_id, score in scored:              # each candidate visited exactly once\n        k = bisect_right(thresholds, score + 1)  # number of thresholds this score passes (score >= threshold)\n        if k == 0:\n            continue\n        pred_diff[0] += 1\n        pred_diff[k] -= 1\n        if cand_id in true_set:\n            tp_diff[0] += 1\n            tp_diff[k] -= 1\n\n    pred_counts = list(accumulate(pred_diff))[:n_thresh]\n    tp_counts = list(accu

In [66]:
"""import pickle

with open('entity_scores_20k.pkl', 'wb') as f:
    pickle.dump(entity_scores, f)

print("Saved entity_scores to disk.")"""

'import pickle\n\nwith open(\'entity_scores_20k.pkl\', \'wb\') as f:\n    pickle.dump(entity_scores, f)\n\nprint("Saved entity_scores to disk.")'

In [67]:
"""import random

random.seed(42)
sample_keys = random.sample(list(entity_scores.keys()), 3000)
entity_scores_small = {k: entity_scores[k] for k in sample_keys}

print("Small sample size:", len(entity_scores_small))"""

'import random\n\nrandom.seed(42)\nsample_keys = random.sample(list(entity_scores.keys()), 3000)\nentity_scores_small = {k: entity_scores[k] for k in sample_keys}\n\nprint("Small sample size:", len(entity_scores_small))'

In [68]:
sample_val_3k = val_source1_keyed_v3.sample(n=3000, random_state=42).reset_index(drop=True)
print("Sampled entities:", len(sample_val_3k))

Sampled entities: 3000


In [72]:
import time
import pickle
from bisect import bisect_right
from itertools import accumulate

thresholds = [70, 75, 80, 85]
n_thresh = len(thresholds)

f05_sums = [0.0] * n_thresh
tp_totals = [0] * n_thresh
pred_totals = [0] * n_thresh
n_entities = 0
true_total = 0

batch_size = 250
start = time.time()

for batch_start in range(0, len(sample_val_3k), batch_size):
    batch = sample_val_3k.iloc[batch_start:batch_start + batch_size]

    for _, row in batch.iterrows():
        entity_id = row['entity_id']
        block_key = row['block_key']
        name1 = row['business_name']
        addr1 = row['business_address']

        candidate_ids = list(get_candidates_v3(block_key))
        scored = score_entity_candidates(name1, addr1, candidate_ids)  # only this entity's pairs, ever

        true_set = parse_ids(val_gt_lookup.get(entity_id))
        true_count = len(true_set)

        n_entities += 1
        true_total += true_count

        pred_diff = [0] * (n_thresh + 1)
        tp_diff = [0] * (n_thresh + 1)

        for cand_id, score in scored:
            k = bisect_right(thresholds, score)
            if k == 0:
                continue
            pred_diff[0] += 1
            pred_diff[k] -= 1
            if cand_id in true_set:
                tp_diff[0] += 1
                tp_diff[k] -= 1
        # `scored` goes out of scope at the next loop iteration — nothing retained

        pred_counts = list(accumulate(pred_diff))[:n_thresh]
        tp_counts = list(accumulate(tp_diff))[:n_thresh]

        for j in range(n_thresh):
            pred_count = pred_counts[j]
            tp = tp_counts[j]

            if true_count == 0:
                f05 = 1.0 if pred_count == 0 else 0.0
            elif pred_count == 0:
                f05 = 0.0
            else:
                precision = tp / pred_count
                recall = tp / true_count
                f05 = 0.0 if (precision == 0 and recall == 0) else (1.25 * precision * recall) / (0.25 * precision + recall)

            f05_sums[j] += f05
            tp_totals[j] += tp
            pred_totals[j] += pred_count

    # Checkpoint compact stats (tiny — a few numbers) after each batch, for safety
    with open('threshold_stats_checkpoint.pkl', 'wb') as f:
        pickle.dump({
            'f05_sums': f05_sums, 'tp_totals': tp_totals, 'pred_totals': pred_totals,
            'n_entities': n_entities, 'true_total': true_total
        }, f)

    print(f"Processed {n_entities}/{len(sample_val_3k)} entities, {time.time()-start:.1f}s elapsed")

elapsed = time.time() - start
print(f"\nDone. Total time: {elapsed:.1f}s")

print(f"\n{'Threshold':>10} {'F0.5':>8} {'Precision':>10} {'Recall':>8}")
for j, t in enumerate(thresholds):
    f05_macro = f05_sums[j] / n_entities
    prec = tp_totals[j] / pred_totals[j] if pred_totals[j] > 0 else 0
    rec = tp_totals[j] / true_total if true_total > 0 else 0
    print(f"{t:>10} {f05_macro:>8.4f} {prec:>10.4f} {rec:>8.4f}")

Processed 250/3000 entities, 138.2s elapsed
Processed 500/3000 entities, 156.1s elapsed
Processed 750/3000 entities, 172.0s elapsed
Processed 1000/3000 entities, 185.0s elapsed
Processed 1250/3000 entities, 195.7s elapsed
Processed 1500/3000 entities, 207.4s elapsed
Processed 1750/3000 entities, 219.5s elapsed
Processed 2000/3000 entities, 232.2s elapsed
Processed 2250/3000 entities, 243.6s elapsed
Processed 2500/3000 entities, 254.3s elapsed
Processed 2750/3000 entities, 265.9s elapsed
Processed 3000/3000 entities, 276.0s elapsed

Done. Total time: 276.0s

 Threshold     F0.5  Precision   Recall
        70   0.4748     0.1799   0.4716
        75   0.4999     0.5474   0.3618
        80   0.4511     0.8259   0.2572
        85   0.3568     0.9128   0.1747


In [69]:
import os 
print(os.path.exists('entity_scores_20k.pkl'))

False


In [70]:
"""import os
size_gb= os.path.getsize('entity_scores_20k.pkl') / (1024**3)
print(f"File size:{size_gb: .2f} GB")"""

'import os\nsize_gb= os.path.getsize(\'entity_scores_20k.pkl\') / (1024**3)\nprint(f"File size:{size_gb: .2f} GB")'

### 5.3 Note — Kernel Restart Wiped entity_scores

After interrupting the threshold sweep due to memory pressure, we attempted to
save `entity_scores` to disk via pickle. This raised `NameError: entity_scores
is not defined`, revealing that the Jupyter kernel had already silently
restarted under memory pressure — losing entity_scores before it could be
persisted.

A leftover 0-byte `entity_scores_20k.pkl` file was found (from the failed save
attempt) and deleted, since it contained no usable data.

Since `entity_scores` no longer exists, we must re-run the RapidFuzz scoring
stage. This time, we checkpoint to disk incrementally (every 2,000 entities,
clearing memory after each save) so a future crash costs at most one chunk of
work, not the full run.

In [71]:
print(len(sample_val_20k))

20000
